# Questão 7 - Apoio ao diagnóstico médico com Raciocínio Baseado em Casos

O sistema recebe os sintomas de um paciente, procura na base os casos mais parecidos e usa esses casos para sugerir um diagnóstico e um tratamento. Depois o diagnóstico pode ser confirmado ou corrigido e o novo caso é guardado na base.

Os casos são fictícios e servem apenas para fins didáticos. O sistema não substitui uma avaliação médica.

In [1]:
import json
from collections import defaultdict
from pathlib import Path

import pandas as pd

ARQUIVO_BASE = Path("base_casos.json")

## 1. Representação dos casos

Cada caso tem três partes: a descrição do problema (sintomas), a solução (diagnóstico e tratamento) e um campo que diz se o caso foi validado.

Os sintomas são guardados com uma intensidade de 0 a 3: 0 ausente, 1 leve, 2 moderado e 3 forte. Usar intensidade em vez de apenas sim ou não ajuda a separar doenças com sintomas parecidos. Gripe e resfriado, por exemplo, podem ter febre, mas na gripe ela costuma ser mais alta.

Sintomas que não aparecem no caso valem 0.

In [2]:
SINTOMAS = [
    "febre", "tosse", "dor_de_cabeca", "dor_no_corpo", "coriza", "dor_de_garganta",
    "nausea", "diarreia", "manchas_na_pele", "falta_de_ar", "perda_de_olfato", "sensibilidade_a_luz",
]

TRATAMENTOS = {
    "Gripe": "Repouso, hidratação e antitérmico. Avaliar antiviral nos casos de risco.",
    "Resfriado": "Repouso, hidratação e lavagem nasal com soro.",
    "Dengue": "Hidratação reforçada e paracetamol. Não usar anti-inflamatórios nem AAS.",
    "Covid-19": "Isolamento, hidratação, antitérmico e acompanhamento da saturação.",
    "Enxaqueca": "Repouso em local escuro e silencioso e analgésico indicado pelo médico.",
    "Gastroenterite": "Soro de reidratação oral e dieta leve.",
}


def caso(id_, diagnostico, **sintomas):
    return {"id": id_, "sintomas": sintomas, "diagnostico": diagnostico,
            "tratamento": TRATAMENTOS[diagnostico], "validado": True}


CASOS_INICIAIS = [
    caso(1, "Gripe", febre=3, tosse=2, dor_de_cabeca=2, dor_no_corpo=3, coriza=1, dor_de_garganta=1),
    caso(2, "Gripe", febre=3, tosse=3, dor_de_cabeca=1, dor_no_corpo=2, dor_de_garganta=2),
    caso(3, "Gripe", febre=2, tosse=2, dor_de_cabeca=2, dor_no_corpo=3, coriza=2),
    caso(4, "Resfriado", febre=1, tosse=1, coriza=3, dor_de_garganta=2),
    caso(5, "Resfriado", tosse=2, coriza=3, dor_de_garganta=1, dor_de_cabeca=1),
    caso(6, "Resfriado", febre=1, tosse=1, coriza=2, dor_de_garganta=3),
    caso(7, "Dengue", febre=3, dor_de_cabeca=3, dor_no_corpo=3, manchas_na_pele=2, nausea=1),
    caso(8, "Dengue", febre=3, dor_de_cabeca=2, dor_no_corpo=3, manchas_na_pele=3, nausea=2),
    caso(9, "Dengue", febre=3, dor_de_cabeca=3, dor_no_corpo=2, nausea=2, manchas_na_pele=1),
    caso(10, "Covid-19", febre=2, tosse=3, falta_de_ar=2, perda_de_olfato=3, dor_no_corpo=1),
    caso(11, "Covid-19", febre=2, tosse=2, falta_de_ar=3, perda_de_olfato=2, dor_de_cabeca=1),
    caso(12, "Covid-19", febre=1, tosse=2, perda_de_olfato=3, dor_de_garganta=1, dor_no_corpo=2),
    caso(13, "Enxaqueca", dor_de_cabeca=3, nausea=2, sensibilidade_a_luz=3),
    caso(14, "Enxaqueca", dor_de_cabeca=3, nausea=1, sensibilidade_a_luz=2),
    caso(15, "Enxaqueca", dor_de_cabeca=2, nausea=3, sensibilidade_a_luz=3),
    caso(16, "Gastroenterite", febre=1, nausea=3, diarreia=3, dor_no_corpo=1),
    caso(17, "Gastroenterite", nausea=2, diarreia=3, dor_de_cabeca=1),
    caso(18, "Gastroenterite", febre=2, nausea=3, diarreia=2),
]

## 2. Base de casos

A base começa com 18 casos de 6 doenças, 3 casos de cada. Ela fica salva em um arquivo JSON para que os casos novos continuem disponíveis nas próximas execuções.

Neste notebook o arquivo é recriado no início para que os resultados abaixo sejam sempre os mesmos.

In [3]:
def salvar_base(casos):
    ARQUIVO_BASE.write_text(json.dumps(casos, ensure_ascii=False, indent=2), encoding="utf-8")


def carregar_base():
    return json.loads(ARQUIVO_BASE.read_text(encoding="utf-8"))


salvar_base(CASOS_INICIAIS)
base = carregar_base()

tabela = pd.DataFrame([{"id": c["id"], "diagnóstico": c["diagnostico"], **c["sintomas"]} for c in base])
tabela = tabela.set_index("id").reindex(columns=["diagnóstico"] + SINTOMAS).fillna(0)
tabela[SINTOMAS] = tabela[SINTOMAS].astype(int)
tabela

,diagnóstico,febre,tosse,dor_de_cabeca,dor_no_corpo,coriza,dor_de_garganta,nausea,diarreia,manchas_na_pele,falta_de_ar,perda_de_olfato,sensibilidade_a_luz
id,,,,,,,,,,,,,
1,Gripe,3,2,2,3,1,1,0,0,0,0,0,0
2,Gripe,3,3,1,2,0,2,0,0,0,0,0,0
3,Gripe,2,2,2,3,2,0,0,0,0,0,0,0
4,Resfriado,1,1,0,0,3,2,0,0,0,0,0,0
5,Resfriado,0,2,1,0,3,1,0,0,0,0,0,0
6,Resfriado,1,1,0,0,2,3,0,0,0,0,0,0
7,Dengue,3,0,3,3,0,0,1,0,2,0,0,0
8,Dengue,3,0,2,3,0,0,2,0,3,0,0,0
9,Dengue,3,0,3,2,0,0,2,0,1,0,0,0


## 3. Cálculo de similaridade

A similaridade entre o caso novo e um caso da base é calculada em duas etapas.

**Similaridade local:** para cada sintoma, compara as duas intensidades. O valor é 1 - |a - b| / 3. Intensidades iguais dão 1 e intensidades opostas (0 e 3) dão 0.

**Similaridade global:** média ponderada das similaridades locais. Só entram na conta os sintomas presentes em pelo menos um dos dois casos. Sem esse cuidado, dois casos quaisquer pareceriam muito semelhantes só por terem vários sintomas ausentes em comum.

Os pesos indicam o quanto cada sintoma ajuda a diferenciar as doenças. Febre e dor de cabeça aparecem em quase todas, então pesam menos. Perda de olfato, manchas na pele e sensibilidade à luz são mais específicas e pesam mais.

In [4]:
PESOS = {
    "febre": 1.0, "tosse": 1.0, "dor_de_cabeca": 1.0, "dor_no_corpo": 1.5, "coriza": 1.5,
    "dor_de_garganta": 1.0, "nausea": 1.5, "diarreia": 2.5, "manchas_na_pele": 3.0,
    "falta_de_ar": 2.5, "perda_de_olfato": 3.0, "sensibilidade_a_luz": 3.0,
}


def sim_local(a, b):
    return 1 - abs(a - b) / 3


def similaridade(s1, s2):
    relevantes = [s for s in SINTOMAS if s1.get(s, 0) or s2.get(s, 0)]
    if not relevantes:
        return 0.0
    soma = sum(PESOS[s] * sim_local(s1.get(s, 0), s2.get(s, 0)) for s in relevantes)
    return soma / sum(PESOS[s] for s in relevantes)

## 4. Ciclo de RBC

**Recuperação (Retrieve):** calcula a similaridade do caso novo com todos os casos da base e devolve os k mais parecidos.

In [5]:
def recuperar(sintomas, base, k=3):
    pontuados = [(similaridade(sintomas, c["sintomas"]), c) for c in base]
    pontuados.sort(key=lambda par: par[0], reverse=True)
    return pontuados[:k]

**Reutilização (Reuse):** cada caso recuperado vota no seu diagnóstico com peso igual à sua similaridade. O diagnóstico com maior soma é o sugerido. A confiança é a fração dos votos que ele recebeu. O tratamento vem do caso mais parecido que tenha esse diagnóstico.

In [6]:
def reutilizar(recuperados):
    votos = defaultdict(float)
    for sim, c in recuperados:
        votos[c["diagnostico"]] += sim
    diagnostico = max(votos, key=votos.get)
    referencia = next(c for _, c in recuperados if c["diagnostico"] == diagnostico)
    return {
        "diagnostico": diagnostico,
        "tratamento": referencia["tratamento"],
        "confianca": votos[diagnostico] / sum(votos.values()),
        "caso_referencia": referencia["id"],
    }

**Revisão (Revise):** o profissional de saúde confirma a sugestão ou informa o diagnóstico correto. Se houver correção, o tratamento também é trocado.

In [7]:
def revisar(sugestao, diagnostico_correto=None, tratamento_correto=None):
    if diagnostico_correto is None or diagnostico_correto == sugestao["diagnostico"]:
        return {"diagnostico": sugestao["diagnostico"], "tratamento": sugestao["tratamento"], "corrigido": False}
    tratamento = tratamento_correto or TRATAMENTOS.get(diagnostico_correto, "A definir pelo médico.")
    return {"diagnostico": diagnostico_correto, "tratamento": tratamento, "corrigido": True}

**Retenção (Retain):** o caso revisado entra na base e o arquivo é salvo. Se já existir um caso quase idêntico com o mesmo diagnóstico, o novo não é guardado, para a base não crescer com casos repetidos.

In [8]:
def reter(sintomas, revisado, base, limite=0.95):
    for c in base:
        if c["diagnostico"] == revisado["diagnostico"] and similaridade(sintomas, c["sintomas"]) >= limite:
            return None
    novo = {
        "id": max(c["id"] for c in base) + 1,
        "sintomas": {s: v for s, v in sintomas.items() if v},
        "diagnostico": revisado["diagnostico"],
        "tratamento": revisado["tratamento"],
        "validado": True,
    }
    base.append(novo)
    salvar_base(base)
    return novo

## 5. Consultas

A função abaixo junta as quatro etapas e mostra o que foi usado em cada uma.

In [9]:
def consultar(sintomas, base, k=3, diagnostico_correto=None, reter_caso=True):
    print("Sintomas informados:", ", ".join(f"{s}={v}" for s, v in sintomas.items()))

    recuperados = recuperar(sintomas, base, k)
    print("\nCasos mais semelhantes:")
    for sim, c in recuperados:
        descricao = ", ".join(f"{s}={v}" for s, v in c["sintomas"].items())
        print(f"  caso {c['id']:>2} | {c['diagnostico']:<14} | similaridade {sim:.2f} | {descricao}")

    sugestao = reutilizar(recuperados)
    print(f"\nDiagnóstico sugerido: {sugestao['diagnostico']} (confiança {sugestao['confianca']:.0%})")
    print("Tratamento sugerido: ", sugestao["tratamento"])

    revisado = revisar(sugestao, diagnostico_correto)
    if revisado["corrigido"]:
        print(f"\nRevisão: diagnóstico corrigido para {revisado['diagnostico']}")
        print("Tratamento após revisão:", revisado["tratamento"])
    else:
        print("\nRevisão: diagnóstico confirmado")

    if not reter_caso:
        print("Retenção: caso não retido.")
        return revisado
    novo = reter(sintomas, revisado, base)
    if novo:
        print(f"Retenção: caso {novo['id']} adicionado. A base tem {len(base)} casos.")
    else:
        print("Retenção: caso muito parecido já existe na base, nada foi adicionado.")
    return revisado

**Consulta 1:** quadro típico de dengue.

In [10]:
_ = consultar({"febre": 3, "dor_de_cabeca": 3, "dor_no_corpo": 3, "manchas_na_pele": 2}, base)

Sintomas informados: febre=3, dor_de_cabeca=3, dor_no_corpo=3, manchas_na_pele=2

Casos mais semelhantes:
  caso  7 | Dengue         | similaridade 0.94 | febre=3, dor_de_cabeca=3, dor_no_corpo=3, manchas_na_pele=2, nausea=1
  caso  8 | Dengue         | similaridade 0.71 | febre=3, dor_de_cabeca=2, dor_no_corpo=3, manchas_na_pele=3, nausea=2
  caso  9 | Dengue         | similaridade 0.69 | febre=3, dor_de_cabeca=3, dor_no_corpo=2, nausea=2, manchas_na_pele=1

Diagnóstico sugerido: Dengue (confiança 100%)
Tratamento sugerido:  Hidratação reforçada e paracetamol. Não usar anti-inflamatórios nem AAS.

Revisão: diagnóstico confirmado
Retenção: caso 19 adicionado. A base tem 19 casos.


**Consulta 2:** tosse, febre e perda de olfato.

In [11]:
_ = consultar({"febre": 2, "tosse": 2, "perda_de_olfato": 2, "dor_de_garganta": 1}, base)

Sintomas informados: febre=2, tosse=2, perda_de_olfato=2, dor_de_garganta=1

Casos mais semelhantes:
  caso 12 | Covid-19       | similaridade 0.69 | febre=1, tosse=2, perda_de_olfato=3, dor_de_garganta=1, dor_no_corpo=2
  caso 11 | Covid-19       | similaridade 0.67 | febre=2, tosse=2, falta_de_ar=3, perda_de_olfato=2, dor_de_cabeca=1
  caso 10 | Covid-19       | similaridade 0.62 | febre=2, tosse=3, falta_de_ar=2, perda_de_olfato=3, dor_no_corpo=1

Diagnóstico sugerido: Covid-19 (confiança 100%)
Tratamento sugerido:  Isolamento, hidratação, antitérmico e acompanhamento da saturação.

Revisão: diagnóstico confirmado
Retenção: caso 20 adicionado. A base tem 20 casos.


**Consulta 3:** poucos sintomas e pouco específicos. Os casos recuperados se dividem entre doenças diferentes e a confiança cai. Como a sugestão é fraca, este caso não é guardado na base.

In [12]:
_ = consultar({"febre": 2, "tosse": 2, "dor_de_cabeca": 1}, base, reter_caso=False)

Sintomas informados: febre=2, tosse=2, dor_de_cabeca=1

Casos mais semelhantes:
  caso 20 | Covid-19       | similaridade 0.62 | febre=2, tosse=2, perda_de_olfato=2, dor_de_garganta=1
  caso  2 | Gripe          | similaridade 0.58 | febre=3, tosse=3, dor_de_cabeca=1, dor_no_corpo=2, dor_de_garganta=2
  caso  1 | Gripe          | similaridade 0.57 | febre=3, tosse=2, dor_de_cabeca=2, dor_no_corpo=3, coriza=1, dor_de_garganta=1

Diagnóstico sugerido: Gripe (confiança 65%)
Tratamento sugerido:  Repouso, hidratação e antitérmico. Avaliar antiviral nos casos de risco.

Revisão: diagnóstico confirmado
Retenção: caso não retido.


**Consulta 4:** caso em que o sistema erra e o médico corrige. O paciente tem febre alta, dor forte no corpo, dor de cabeça e um pouco de tosse, ainda sem manchas na pele. O quadro parece gripe, mas o exame confirmou dengue.

In [13]:
paciente = {"febre": 3, "dor_no_corpo": 3, "dor_de_cabeca": 2, "tosse": 1}
_ = consultar(paciente, base, diagnostico_correto="Dengue")

Sintomas informados: febre=3, dor_no_corpo=3, dor_de_cabeca=2, tosse=1

Casos mais semelhantes:
  caso  1 | Gripe          | similaridade 0.83 | febre=3, tosse=2, dor_de_cabeca=2, dor_no_corpo=3, coriza=1, dor_de_garganta=1
  caso  3 | Gripe          | similaridade 0.72 | febre=2, tosse=2, dor_de_cabeca=2, dor_no_corpo=3, coriza=2
  caso  7 | Dengue         | similaridade 0.65 | febre=3, dor_de_cabeca=3, dor_no_corpo=3, manchas_na_pele=2, nausea=1

Diagnóstico sugerido: Gripe (confiança 71%)
Tratamento sugerido:  Repouso, hidratação e antitérmico. Avaliar antiviral nos casos de risco.

Revisão: diagnóstico corrigido para Dengue
Tratamento após revisão: Hidratação reforçada e paracetamol. Não usar anti-inflamatórios nem AAS.
Retenção: caso 21 adicionado. A base tem 21 casos.


**Consulta 5:** um paciente com quadro parecido com o da consulta 4 chega depois. Agora o caso retido é recuperado e o sistema acerta. Isso mostra o aprendizado do RBC: ele melhora guardando casos, sem treinar nenhum modelo.

In [14]:
_ = consultar({"febre": 3, "dor_no_corpo": 3, "dor_de_cabeca": 3, "tosse": 1}, base)

Sintomas informados: febre=3, dor_no_corpo=3, dor_de_cabeca=3, tosse=1

Casos mais semelhantes:
  caso 21 | Dengue         | similaridade 0.93 | febre=3, dor_no_corpo=3, dor_de_cabeca=2, tosse=1
  caso  1 | Gripe          | similaridade 0.79 | febre=3, tosse=2, dor_de_cabeca=2, dor_no_corpo=3, coriza=1, dor_de_garganta=1
  caso 19 | Dengue         | similaridade 0.69 | febre=3, dor_de_cabeca=3, dor_no_corpo=3, manchas_na_pele=2

Diagnóstico sugerido: Dengue (confiança 67%)
Tratamento sugerido:  Hidratação reforçada e paracetamol. Não usar anti-inflamatórios nem AAS.

Revisão: diagnóstico confirmado
Retenção: caso 22 adicionado. A base tem 22 casos.


## 6. Consulta interativa

A função abaixo faz a mesma consulta perguntando os sintomas pelo teclado. Ela não é chamada automaticamente para o notebook poder ser executado de uma vez. Para usar, basta executar `consulta_interativa(base)` em uma célula.

In [15]:
def consulta_interativa(base):
    print("Informe a intensidade de cada sintoma (0 ausente, 1 leve, 2 moderado, 3 forte).")
    sintomas = {}
    for s in SINTOMAS:
        valor = input(f"{s}: ").strip()
        if valor in ("1", "2", "3"):
            sintomas[s] = int(valor)
    sugestao = reutilizar(recuperar(sintomas, base))
    print("Sugestão:", sugestao["diagnostico"])
    resposta = input("Enter para confirmar ou digite o diagnóstico correto: ").strip()
    return consultar(sintomas, base, diagnostico_correto=resposta or None)

## 7. Análise dos resultados

Para medir a qualidade da recuperação foi usado o teste deixa-um-de-fora nos 18 casos iniciais: cada caso é retirado da base, consultado como se fosse novo e o diagnóstico sugerido é comparado com o real.

In [16]:
def deixa_um_de_fora(casos, k):
    acertos = 0
    for c in casos:
        resto = [o for o in casos if o["id"] != c["id"]]
        acertos += reutilizar(recuperar(c["sintomas"], resto, k))["diagnostico"] == c["diagnostico"]
    return acertos / len(casos)


pd.DataFrame([{"k": k, "acurácia": f"{deixa_um_de_fora(CASOS_INICIAIS, k):.0%}"} for k in (1, 2, 3, 5)])

,k,acurácia
0,1,100%
1,2,100%
2,3,100%
3,5,72%


In [17]:
print("Casos na base ao final:", len(base))
pd.DataFrame([{"id": c["id"], "diagnóstico": c["diagnostico"], **c["sintomas"]} for c in base[18:]]).fillna(0)

Casos na base ao final: 22


,id,diagnóstico,febre,dor_de_cabeca,dor_no_corpo,manchas_na_pele,tosse,perda_de_olfato,dor_de_garganta
0,19,Dengue,3,3.0,3.0,2.0,0.0,0.0,0.0
1,20,Covid-19,2,0.0,0.0,0.0,2.0,2.0,1.0
2,21,Dengue,3,2.0,3.0,0.0,1.0,0.0,0.0
3,22,Dengue,3,3.0,3.0,0.0,1.0,0.0,0.0


Nas consultas com sintomas específicos (1 e 2) os três casos recuperados são da mesma doença e a confiança é alta. O peso maior de manchas na pele e de perda de olfato foi o que mais pesou nesses resultados.

Na consulta 3 os sintomas são comuns a várias doenças. O sistema ainda dá uma sugestão, mas a confiança menor avisa que ela é fraca. Esse é um comportamento desejado: o sistema deixa claro quando os casos recuperados não concordam.

As consultas 4 e 5 mostram o ciclo completo. O erro da consulta 4 foi corrigido na revisão, o caso foi retido e passou a ser usado na consulta seguinte.

O teste deixa-um-de-fora acertou todos os casos com k até 3, mas isso precisa ser lido com cuidado. A base é pequena e os casos de cada doença foram criados com sintomas bem característicos. Com pacientes reais haveria mais sobreposição entre as doenças. Com k = 5 a acurácia cai porque cada doença tem só 3 casos: ao retirar um, sobram 2 da doença certa e os outros 3 vizinhos são sempre de outras doenças. Por isso o sistema usa k = 3.

As principais limitações são o tamanho da base, os pesos definidos manualmente e a falta de outras informações do paciente, como idade, duração dos sintomas e resultados de exames. O sistema também não faz adaptação da solução: ele copia o tratamento do caso mais parecido.